# Sistemas Inteligentes I
## Búsqueda adversarial: poda Alfa–Beta

**Autor:** Jairo I. Vélez B.

---


# 1. Punto de partida: el problema de Minimax

Minimax supone que:

- MAX intenta maximizar;
- MIN intenta minimizar;
- ambos jugadores actúan racionalmente.

El problema es que, para garantizar su decisión, Minimax puede explorar aproximadamente:

$$O(b^m)$$

nodos.

Sin embargo, algunas ramas pueden resultar irrelevantes.

La idea central de Alfa–Beta es:

> **dejar de explorar una rama cuando ya sabemos que no puede mejorar la decisión de un jugador.**

La poda no cambia el resultado de Minimax.  
Solo intenta obtenerlo explorando menos nodos.

# 2. Recordatorio: un árbol de juego

Utilizaremos inicialmente el mismo tipo de representación:

```text
                    A  (MAX)
              /         |         \
          B (MIN)    C (MIN)    D (MIN)
          / | \       / | \       / | \
         3  5  2     9  1  4     6  7  8
```

Minimax calcula:

- `B = 2`
- `C = 1`
- `D = 6`

y finalmente:

$$A=\max(2,1,6)=6$$

In [ ]:
arbol = {
    "A": ["B", "C", "D"],
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
    "D": ["D1", "D2", "D3"],
}

utilidades = {
    "B1": 3, "B2": 5, "B3": 2,
    "C1": 9, "C2": 1, "C3": 4,
    "D1": 6, "D2": 7, "D3": 8,
}

arbol, utilidades

# 3. ¿Qué representan alfa y beta?

Durante la búsqueda mantenemos dos límites.

### Alfa — $\alpha$

Es el mejor valor que **MAX puede garantizar hasta el momento**.

Inicialmente:

$$\alpha=-\infty$$

### Beta — $\beta$

Es el mejor valor que **MIN puede garantizar hasta el momento**.

Inicialmente:

$$\beta=+\infty$$

Durante la búsqueda:

- MAX actualiza $\alpha$;
- MIN actualiza $\beta$.

Cuando ocurre:

$$\boxed{\alpha \geq \beta}$$

podemos realizar una **poda**.

# 4. Intuición de una poda

Suponga que MAX ya dispone de una alternativa con valor `6`.

Ahora explora otra rama cuyo turno pertenece a MIN.

Si MIN encuentra dentro de esa rama una opción con valor `4`, sabemos que podrá forzar:

$$valor \leq 4$$

MAX ya dispone de `6`, por lo que nunca elegirá una alternativa que termine en
`4` o menos.

Por tanto:

> **el resto de esa rama ya no puede cambiar la decisión de MAX.**

Podemos dejar de explorarla.

# 5. Implementación de Alfa–Beta

La estructura es muy similar a Minimax.

La diferencia está en que conservamos y actualizamos los límites
$\alpha$ y $\beta$.

In [ ]:
from math import inf

def alfa_beta(nodo, es_max, arbol, utilidades, alfa=-inf, beta=inf):
    if nodo in utilidades:
        return utilidades[nodo]

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor = max(
                valor,
                alfa_beta(hijo, False, arbol, utilidades, alfa, beta)
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor = min(
                valor,
                alfa_beta(hijo, True, arbol, utilidades, alfa, beta)
            )

            beta = min(beta, valor)

            if alfa >= beta:
                break

        return valor


alfa_beta("A", True, arbol, utilidades)

# 6. Comparar Alfa–Beta con Minimax

Primero implementaremos una versión sencilla de Minimax.

In [ ]:
def minimax(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo]

    valores = [
        minimax(hijo, not es_max, arbol, utilidades)
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


print("Minimax   :", minimax("A", True, arbol, utilidades))
print("Alfa-Beta :", alfa_beta("A", True, arbol, utilidades))

# 7. Alfa–Beta paso a paso

Ahora imprimiremos:

- nodo visitado;
- jugador;
- valor de $\alpha$;
- valor de $\beta$;
- momento en el que se produce una poda.

In [ ]:
def alfa_beta_debug(
    nodo,
    es_max,
    arbol,
    utilidades,
    alfa=-inf,
    beta=inf,
    profundidad=0
):
    sangria = "    " * profundidad
    jugador = "MAX" if es_max else "MIN"

    if nodo in utilidades:
        print(
            f"{sangria}{nodo}: terminal = {utilidades[nodo]} "
            f"[α={alfa}, β={beta}]"
        )
        return utilidades[nodo]

    print(
        f"{sangria}{nodo}: {jugador} "
        f"[α={alfa}, β={beta}]"
    )

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor_hijo = alfa_beta_debug(
                hijo,
                False,
                arbol,
                utilidades,
                alfa,
                beta,
                profundidad + 1
            )

            valor = max(valor, valor_hijo)
            alfa = max(alfa, valor)

            print(
                f"{sangria}  después de {hijo}: "
                f"valor={valor}, α={alfa}, β={beta}"
            )

            if alfa >= beta:
                print(
                    f"{sangria}  PODA en {nodo}: "
                    f"α={alfa} >= β={beta}"
                )
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor_hijo = alfa_beta_debug(
                hijo,
                True,
                arbol,
                utilidades,
                alfa,
                beta,
                profundidad + 1
            )

            valor = min(valor, valor_hijo)
            beta = min(beta, valor)

            print(
                f"{sangria}  después de {hijo}: "
                f"valor={valor}, α={alfa}, β={beta}"
            )

            if alfa >= beta:
                print(
                    f"{sangria}  PODA en {nodo}: "
                    f"α={alfa} >= β={beta}"
                )
                break

        return valor


alfa_beta_debug("A", True, arbol, utilidades)

# 8. Un ejemplo diseñado para observar podas

El orden de los valores del árbol anterior no siempre produce una poda muy visible.

Usaremos ahora este árbol:

```text
                         A (MAX)
                    /             \
               B (MIN)           C (MIN)
              /      \           /      \
          D(MAX)   E(MAX)    F(MAX)    G(MAX)
           3  5     6  9      1  2      0 -1
```

La exploración se realiza de izquierda a derecha.

In [ ]:
arbol_poda = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G1", "G2"],
}

utilidades_poda = {
    "D1": 3, "D2": 5,
    "E1": 6, "E2": 9,
    "F1": 1, "F2": 2,
    "G1": 0, "G2": -1,
}

print("Minimax:", minimax("A", True, arbol_poda, utilidades_poda))
print()
alfa_beta_debug("A", True, arbol_poda, utilidades_poda)

# 9. Medir el ahorro de exploración

Para comparar los algoritmos contabilizaremos:

- nodos visitados;
- hojas evaluadas;
- podas realizadas.

In [ ]:
def minimax_contando(nodo, es_max, arbol, utilidades, estadisticas):
    estadisticas["visitados"] += 1

    if nodo in utilidades:
        estadisticas["hojas"] += 1
        return utilidades[nodo]

    valores = [
        minimax_contando(
            hijo,
            not es_max,
            arbol,
            utilidades,
            estadisticas
        )
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


def alfa_beta_contando(
    nodo,
    es_max,
    arbol,
    utilidades,
    estadisticas,
    alfa=-inf,
    beta=inf
):
    estadisticas["visitados"] += 1

    if nodo in utilidades:
        estadisticas["hojas"] += 1
        return utilidades[nodo]

    if es_max:
        valor = -inf

        for hijo in arbol[nodo]:
            valor = max(
                valor,
                alfa_beta_contando(
                    hijo,
                    False,
                    arbol,
                    utilidades,
                    estadisticas,
                    alfa,
                    beta
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                estadisticas["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for hijo in arbol[nodo]:
            valor = min(
                valor,
                alfa_beta_contando(
                    hijo,
                    True,
                    arbol,
                    utilidades,
                    estadisticas,
                    alfa,
                    beta
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                estadisticas["podas"] += 1
                break

        return valor


stats_minimax = {"visitados": 0, "hojas": 0}
stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

valor_mm = minimax_contando(
    "A", True, arbol_poda, utilidades_poda, stats_minimax
)

valor_ab = alfa_beta_contando(
    "A", True, arbol_poda, utilidades_poda, stats_ab
)

print("Valor Minimax:", valor_mm)
print("Valor Alfa-Beta:", valor_ab)
print()
print("Minimax:", stats_minimax)
print("Alfa-Beta:", stats_ab)

### Preguntas de análisis

Resultados obtenidos al ejecutar la celda anterior:

| Algoritmo | Valor | Nodos visitados | Hojas evaluadas | Podas |
|-----------|:-----:|:---------------:|:---------------:|:-----:|
| Minimax   | 5     | 15              | 8               | —     |
| Alfa–Beta | 5     | 11              | 5               | 2     |

**1. ¿Ambos algoritmos producen el mismo valor?**

Sí, los dos devuelven **5**. Alfa–Beta es una optimización de Minimax: solo descarta ramas que no pueden influir en la decisión de la raíz, así que siempre llega al mismo valor minimax.

**2. ¿Cuántas hojas evita evaluar Alfa–Beta?**

Evita evaluar **3 hojas** (8 − 5 = 3): `E2`, `G1` y `G2`. En total visita 4 nodos menos (15 − 11), porque tampoco entra al nodo interno `G`. Hubo 2 podas:

- **Poda en `E`** (nodo MAX dentro de `B`): después de `D`, MIN en `B` ya tiene β = 5. Al evaluar `E1 = 6`, MAX en `E` tiene α = 6 ≥ β = 5. Como MAX obtendría al menos 6 en `E`, MIN nunca elegiría `E` (ya tiene 5 con `D`), así que `E2` no se evalúa.
- **Poda en `C`** (nodo MIN): la raíz ya tiene α = 5 gracias a `B`. En `C`, la rama `F` vale 2, así que β = 2 y α = 5 ≥ β = 2. MIN puede forzar un valor ≤ 2 en `C`, y MAX ya tiene 5 con `B`, así que el subárbol `G` (`G1`, `G2`) no se explora.

**3. ¿Qué información permite justificar una poda?**

Los límites **α** (lo mínimo que MAX ya tiene asegurado en el camino) y **β** (lo máximo que MIN ya tiene asegurado). Se poda cuando **α ≥ β**. En ese punto, el valor del nodo actual ya queda fuera del intervalo que algún ancestro aceptaría, porque ese ancestro tiene una alternativa igual o mejor. Explorar los hijos que faltan solo podría empeorar el nodo para ese ancestro, así que el valor en la raíz no cambia.

**4. ¿Podar significa que la rama sea necesariamente mala?**

No. Podar significa que la rama es **irrelevante para la decisión**, no que sea mala. Por ejemplo, en `E` la hoja podada `E2 = 9` era mejor para MAX que todo lo demás. Aun así se podó, porque MIN nunca dejaría que el juego llegara a `E`. Una rama se poda porque el oponente (o el propio jugador) tiene una alternativa mejor antes de llegar a ella, no por el valor de la rama en sí.

**5. ¿Podría una rama podada contener valores muy altos o muy bajos?**

Sí, puede tener cualquier valor, extremo o no, y el resultado no cambia:

- En la poda de `E` se perdió `E2 = 9`, el valor más alto del árbol. Aunque fuera +∞, `E` valdría al menos 6 > 5 y MIN seguiría eligiendo `D`.
- En la poda de `C` se perdió el subárbol `G`. Aunque `G` valiera +∞, `C = min(2, G) ≤ 2`, y si valiera −∞, `C` sería aún peor para MAX. En los dos casos MAX elige `B` con valor 5.

La poda es válida precisamente porque la conclusión no depende de los valores que se dejan sin evaluar.

# 10. El orden de exploración importa

Alfa–Beta es especialmente eficaz cuando primero examinamos las jugadas más prometedoras.

En el mejor caso, su complejidad puede aproximarse a:

$$O(b^{m/2})$$

en lugar de:

$$O(b^m)$$

Esto significa que, con un buen ordenamiento, puede ser posible explorar
aproximadamente el doble de profundidad usando recursos comparables.

Sin embargo:

> **Alfa–Beta sigue siendo correcto independientemente del orden.  
> El orden afecta cuánto poda, no el valor final.**

## 10.1 Comparar dos órdenes del mismo árbol

Crearemos dos versiones:

- una con un orden favorable;
- otra con un orden menos favorable.

Los valores terminales son exactamente los mismos.

In [ ]:
arbol_buen_orden = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D2", "D1"],
    "E": ["E2", "E1"],
    "F": ["F2", "F1"],
    "G": ["G1", "G2"],
}

arbol_mal_orden = {
    "A": ["C", "B"],
    "B": ["E", "D"],
    "C": ["G", "F"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G2", "G1"],
}

def medir_alfa_beta(arbol):
    stats = {"visitados": 0, "hojas": 0, "podas": 0}
    valor = alfa_beta_contando(
        "A",
        True,
        arbol,
        utilidades_poda,
        stats
    )
    return valor, stats


print("Orden 1:", medir_alfa_beta(arbol_buen_orden))
print("Orden 2:", medir_alfa_beta(arbol_mal_orden))

### Preguntas de análisis

Resultados obtenidos al ejecutar la celda anterior:

| Orden | Valor | Nodos visitados | Hojas evaluadas | Podas |
|-------|:-----:|:---------------:|:---------------:|:-----:|
| Orden 1 (favorable)       | 5 | 11 | 5 | 2 |
| Orden 2 (menos favorable) | 5 | 14 | 7 | 1 |

**1. ¿Cambió el valor final?**

No. Los dos órdenes devuelven **5**, igual que Minimax. Las hojas tienen los mismos valores, solo cambia el orden en que se exploran, y Alfa–Beta es correcto sin importar ese orden.

**2. ¿Cambió el número de nodos visitados?**

Sí. El orden 1 visita **11** nodos (5 hojas, 2 podas) y el orden 2 visita **14** (7 hojas, 1 poda). El orden 2 explora primero `C`, que es la peor rama para MAX:

- En `C` aparece primero `G = max(-1, 0) = 0`, así que la raíz queda con α = 0, un límite bajo.
- Al pasar a `B`, se explora primero `E`. Con α = 0 y β = +∞ no hay poda posible, así que se evalúan `E1 = 6` y `E2 = 9`.
- Después `D` vale 5, y `B = 5`. Casi todo `B` se exploró completo porque el α de la raíz (0) era demasiado pobre para descartar nada.

En el orden 1, MAX encuentra primero `B = 5` (la mejor jugada). Con α = 5 en la raíz, al ver `F = 2` dentro de `C` ya puede descartar todo el subárbol `G`. Dentro de `B`, al explorar primero `E2 = 9` se poda `E1` inmediatamente, porque MIN ya tiene 5 con `D`.

**3. ¿Por qué conocer primero una buena jugada ayuda a podar?**

Porque una buena jugada ajusta los límites desde el principio: α sube rápido en los nodos MAX y β baja rápido en los nodos MIN. Mientras más estrecho sea el intervalo [α, β], antes se cumple α ≥ β en las ramas siguientes. Si la mejor jugada aparece al final, los límites se mantienen amplios casi toda la búsqueda y hay que revisar las alternativas malas completas antes de poder descartarlas. En el mejor caso (siempre se examina primero la mejor jugada) la complejidad se acerca a $O(b^{m/2})$.

**4. ¿Cómo podría un programa real ordenar las jugadas antes de examinarlas?**

- **Heurísticas del dominio**: en ajedrez, examinar primero capturas, jaques y promociones. En Tres en raya, primero el centro, luego las esquinas y al final los bordes (se prueba en la sección 13).
- **Función de evaluación barata**: evaluar cada hijo con una heurística rápida y ordenarlos de mejor a peor antes de la búsqueda profunda.
- **Profundización iterativa**: buscar a profundidad 1, 2, 3, ... y usar la mejor jugada de la iteración anterior como la primera de la siguiente.
- **Tablas de transposición**: guardar la mejor jugada ya calculada para cada posición y probarla primero si la posición vuelve a aparecer.
- **Killer moves / history heuristic**: dar prioridad a las jugadas que produjeron podas en otras ramas del mismo nivel.

# 11. Caso aplicado: juego de las piedras

Retomaremos el juego:

- hay una pila de piedras;
- cada jugador puede retirar `1`, `2` o `3`;
- quien retira la última piedra gana.

Compararemos Minimax y Alfa–Beta sobre el mismo juego.

In [ ]:
MOVIMIENTOS = (1, 2, 3)

def movimientos_validos(piedras):
    return [m for m in MOVIMIENTOS if m <= piedras]


def minimax_piedras_contando(piedras, turno_max, stats):
    stats["visitados"] += 1

    if piedras == 0:
        stats["hojas"] += 1
        return -1 if turno_max else 1

    valores = [
        minimax_piedras_contando(
            piedras - retirar,
            not turno_max,
            stats
        )
        for retirar in movimientos_validos(piedras)
    ]

    return max(valores) if turno_max else min(valores)


def alfa_beta_piedras(
    piedras,
    turno_max,
    stats,
    alfa=-inf,
    beta=inf
):
    stats["visitados"] += 1

    if piedras == 0:
        stats["hojas"] += 1
        return -1 if turno_max else 1

    if turno_max:
        valor = -inf

        for retirar in movimientos_validos(piedras):
            valor = max(
                valor,
                alfa_beta_piedras(
                    piedras - retirar,
                    False,
                    stats,
                    alfa,
                    beta
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                stats["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for retirar in movimientos_validos(piedras):
            valor = min(
                valor,
                alfa_beta_piedras(
                    piedras - retirar,
                    True,
                    stats,
                    alfa,
                    beta
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                stats["podas"] += 1
                break

        return valor

## 11.1 Comparación experimental

In [ ]:
for piedras in [6, 8, 10, 12]:
    stats_mm = {"visitados": 0, "hojas": 0}
    stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

    valor_mm = minimax_piedras_contando(
        piedras, True, stats_mm
    )

    valor_ab = alfa_beta_piedras(
        piedras, True, stats_ab
    )

    print(f"\n{piedras} piedras")
    print("  Minimax   :", valor_mm, stats_mm)
    print("  Alfa-Beta :", valor_ab, stats_ab)

### Observaciones

| Piedras | Valor | Nodos Minimax | Nodos Alfa–Beta | Ahorro |
|:-------:|:-----:|:-------------:|:---------------:|:------:|
| 6  |  1 | 52    | 45  | 13,5 % |
| 8  | -1 | 177   | 134 | 24,3 % |
| 10 |  1 | 600   | 329 | 45,2 % |
| 12 | -1 | 2031  | 987 | 51,4 % |

- Los dos algoritmos dan siempre el mismo valor.
- Con 8 y 12 piedras (múltiplos de 4) el valor es −1: MAX pierde si MIN juega bien, porque haga lo que haga, MIN puede retirar lo necesario para dejar otra vez un múltiplo de 4.
- El ahorro crece con el tamaño del problema: con 6 piedras Alfa–Beta visita un 13 % menos de nodos y con 12, un 51 % menos. Con más piedras el árbol crece exponencialmente y cada poda elimina un subárbol más grande.

# 12. Obtener la mejor jugada con Alfa–Beta

En una aplicación real necesitamos devolver una **acción**, no solo el valor.

In [ ]:
def mejor_jugada_alfa_beta_piedras(piedras):
    mejor_valor = -inf
    mejor_movimiento = None
    alfa = -inf
    beta = inf

    for retirar in movimientos_validos(piedras):
        stats = {"visitados": 0, "hojas": 0, "podas": 0}

        valor = alfa_beta_piedras(
            piedras - retirar,
            False,
            stats,
            alfa,
            beta
        )

        if valor > mejor_valor:
            mejor_valor = valor
            mejor_movimiento = retirar

        alfa = max(alfa, mejor_valor)

    return mejor_movimiento, mejor_valor


for piedras in range(1, 11):
    movimiento, valor = mejor_jugada_alfa_beta_piedras(piedras)

    print(
        f"{piedras:2d} piedras -> "
        f"retirar {movimiento}, valor {valor}"
    )

# 13. Taller: Alfa–Beta para Tres en raya

Se reutiliza la representación del Taller 3 (Minimax):

- el tablero es una tupla de nueve posiciones (`"X"`, `"O"` o `" "`);
- `X` es MAX y `O` es MIN;
- victoria de `X`: `+1`, empate: `0`, victoria de `O`: `-1`.

La función `alfa_beta_tictactoe` tiene la misma estructura que `alfa_beta` de la sección 5. Además recibe un diccionario `stats` opcional para contar nodos, hojas y podas.

In [ ]:
"""
Alfa-Beta para Tres en raya

    0 | 1 | 2
    ---------
    3 | 4 | 5
    ---------
    6 | 7 | 8

X = MAX (utilidad +1)
O = MIN (utilidad -1)
empate = 0
"""

from math import inf

LINEAS_GANADORAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),  # filas
    (0, 3, 6), (1, 4, 7), (2, 5, 8),  # columnas
    (0, 4, 8), (2, 4, 6),             # diagonales
]


def acciones(tablero):
    """Índices de las casillas vacías."""
    return [i for i, casilla in enumerate(tablero) if casilla == " "]


def resultado(tablero, accion, jugador):
    """Nuevo tablero con `jugador` en la posición `accion` (no modifica el original)."""
    nuevo = list(tablero)
    nuevo[accion] = jugador
    return tuple(nuevo)


def ganador(tablero):
    """Devuelve 'X', 'O' o None si nadie ha ganado."""
    for a, b, c in LINEAS_GANADORAS:
        if tablero[a] == tablero[b] == tablero[c] != " ":
            return tablero[a]
    return None


def terminal(tablero):
    """El juego termina si hay ganador o si no quedan casillas vacías."""
    return ganador(tablero) is not None or " " not in tablero


def utilidad(tablero):
    """+1 si ganó X, -1 si ganó O, 0 si hubo empate."""
    g = ganador(tablero)
    if g == "X":
        return 1
    if g == "O":
        return -1
    return 0


def alfa_beta_tictactoe(
    tablero,
    es_max,
    alfa=-inf,
    beta=inf,
    stats=None
):
    if stats is not None:
        stats["visitados"] += 1

    if terminal(tablero):
        if stats is not None:
            stats["hojas"] += 1
        return utilidad(tablero)

    if es_max:
        valor = -inf

        for accion in acciones(tablero):
            valor = max(
                valor,
                alfa_beta_tictactoe(
                    resultado(tablero, accion, "X"),
                    False,
                    alfa,
                    beta,
                    stats
                )
            )

            alfa = max(alfa, valor)

            if alfa >= beta:
                if stats is not None:
                    stats["podas"] += 1
                break

        return valor

    else:
        valor = inf

        for accion in acciones(tablero):
            valor = min(
                valor,
                alfa_beta_tictactoe(
                    resultado(tablero, accion, "O"),
                    True,
                    alfa,
                    beta,
                    stats
                )
            )

            beta = min(beta, valor)

            if alfa >= beta:
                if stats is not None:
                    stats["podas"] += 1
                break

        return valor

## 13.1 Comparación con Minimax

Se implementa un Minimax con contadores para comparar ambos algoritmos sobre los mismos tableros. El turno se deduce del tablero: si hay tantas `X` como `O`, le toca a `X`.

In [ ]:
def minimax_tictactoe_contando(tablero, es_max, stats):
    stats["visitados"] += 1

    if terminal(tablero):
        stats["hojas"] += 1
        return utilidad(tablero)

    jugador = "X" if es_max else "O"

    valores = [
        minimax_tictactoe_contando(
            resultado(tablero, accion, jugador),
            not es_max,
            stats
        )
        for accion in acciones(tablero)
    ]

    return max(valores) if es_max else min(valores)


tableros_prueba = {
    "Vacío (turno X)": (
        " ", " ", " ",
        " ", " ", " ",
        " ", " ", " ",
    ),
    "X en el centro (turno O)": (
        " ", " ", " ",
        " ", "X", " ",
        " ", " ", " ",
    ),
    "Enunciado Taller 3 (turno X)": (
        "X", "O", " ",
        " ", "X", " ",
        "O", " ", " ",
    ),
    "O amenaza fila media (turno X)": (
        "X", " ", " ",
        "O", "O", " ",
        "X", " ", " ",
    ),
}

for nombre, tablero in tableros_prueba.items():
    es_max = tablero.count("X") == tablero.count("O")

    stats_mm = {"visitados": 0, "hojas": 0}
    stats_ab = {"visitados": 0, "hojas": 0, "podas": 0}

    valor_mm = minimax_tictactoe_contando(tablero, es_max, stats_mm)
    valor_ab = alfa_beta_tictactoe(tablero, es_max, stats=stats_ab)

    ahorro = 1 - stats_ab["visitados"] / stats_mm["visitados"]

    print(nombre)
    print("  Minimax   :", valor_mm, stats_mm)
    print("  Alfa-Beta :", valor_ab, stats_ab)
    print(f"  Ahorro en nodos visitados: {ahorro:.1%}\n")

## 13.2 Mejor jugada

Como en la sección 12, a la raíz no le basta con el valor: necesitamos la **acción**. Se prueba cada jugada, se evalúa con Alfa–Beta y se conserva la mejor, actualizando α (si juega `X`) o β (si juega `O`) para que las siguientes jugadas también se beneficien de la poda.

In [ ]:
def mostrar(tablero):
    filas = [
        " | ".join(tablero[i:i + 3])
        for i in range(0, 9, 3)
    ]
    print("\n---------\n".join(filas))


def mejor_jugada_tictactoe(tablero):
    """Devuelve (acción, valor) para el jugador al que le toca mover."""
    es_max = tablero.count("X") == tablero.count("O")
    jugador = "X" if es_max else "O"

    mejor_accion = None
    mejor_valor = -inf if es_max else inf
    alfa, beta = -inf, inf

    for accion in acciones(tablero):
        valor = alfa_beta_tictactoe(
            resultado(tablero, accion, jugador),
            not es_max,
            alfa,
            beta
        )

        if es_max and valor > mejor_valor:
            mejor_valor, mejor_accion = valor, accion
            alfa = max(alfa, valor)
        elif not es_max and valor < mejor_valor:
            mejor_valor, mejor_accion = valor, accion
            beta = min(beta, valor)

    return mejor_accion, mejor_valor


for nombre, tablero in tableros_prueba.items():
    accion, valor = mejor_jugada_tictactoe(tablero)
    print(nombre)
    mostrar(tablero)
    print(f"-> mejor jugada: casilla {accion}, valor {valor}\n")

## 13.3 Partida completa: Alfa–Beta contra Alfa–Beta

Si los dos jugadores juegan de forma óptima, Tres en raya termina siempre en empate.

In [ ]:
tablero = (" ",) * 9

while not terminal(tablero):
    jugador = "X" if tablero.count("X") == tablero.count("O") else "O"
    accion, _ = mejor_jugada_tictactoe(tablero)
    tablero = resultado(tablero, accion, jugador)
    print(f"{jugador} juega en la casilla {accion}")

print()
mostrar(tablero)
print("\nUtilidad final:", utilidad(tablero))

## 13.4 Efecto del orden de las jugadas

Siguiendo la sección 10, se prueba ordenar las jugadas con una heurística sencilla: primero el **centro** (participa en 4 líneas), luego las **esquinas** (3 líneas) y al final los **bordes** (2 líneas).

In [ ]:
PRIORIDAD = [4, 0, 2, 6, 8, 1, 3, 5, 7]  # centro, esquinas, bordes


def acciones_ordenadas(tablero):
    return [i for i in PRIORIDAD if tablero[i] == " "]


def alfa_beta_tictactoe_ordenado(
    tablero,
    es_max,
    stats,
    alfa=-inf,
    beta=inf
):
    stats["visitados"] += 1

    if terminal(tablero):
        stats["hojas"] += 1
        return utilidad(tablero)

    jugador = "X" if es_max else "O"
    valor = -inf if es_max else inf

    for accion in acciones_ordenadas(tablero):
        valor_hijo = alfa_beta_tictactoe_ordenado(
            resultado(tablero, accion, jugador),
            not es_max,
            stats,
            alfa,
            beta
        )

        if es_max:
            valor = max(valor, valor_hijo)
            alfa = max(alfa, valor)
        else:
            valor = min(valor, valor_hijo)
            beta = min(beta, valor)

        if alfa >= beta:
            stats["podas"] += 1
            break

    return valor


vacio = (" ",) * 9

stats_normal = {"visitados": 0, "hojas": 0, "podas": 0}
stats_ordenado = {"visitados": 0, "hojas": 0, "podas": 0}

print("Orden 0..8        :", alfa_beta_tictactoe(vacio, True, stats=stats_normal), stats_normal)
print("Centro/esquinas   :", alfa_beta_tictactoe_ordenado(vacio, True, stats_ordenado), stats_ordenado)

## 13.5 Análisis de resultados

| Tablero | Valor | Nodos Minimax | Nodos Alfa–Beta | Ahorro |
|---------|:-----:|:-------------:|:---------------:|:------:|
| Vacío (turno X)                | 0 | 549 946 | 18 297 | 96,7 % |
| X en el centro (turno O)       | 0 | 55 505  | 2 316  | 95,8 % |
| Enunciado Taller 3 (turno X)   | 1 | 182     | 97     | 46,7 % |
| O amenaza fila media (turno X) | 0 | 206     | 95     | 53,9 % |

1. **Mismo resultado.** En todos los tableros Alfa–Beta devuelve el mismo valor que Minimax. La poda solo descarta ramas que no pueden cambiar la decisión.
2. **Ahorro enorme en árboles grandes.** Con el tablero vacío, Minimax visita casi 550 000 nodos y Alfa–Beta unos 18 000, un 96,7 % menos. En tableros casi llenos el árbol es pequeño y el ahorro baja a cerca del 50 %.
3. **El valor del tablero vacío es 0.** Si los dos juegan bien, el resultado es empate, como muestra la partida de la sección 13.3.
4. **Las jugadas elegidas tienen sentido:**
   - En el tablero con `O` en 3 y 4, `X` bloquea en la casilla 5.
   - En el tablero del enunciado del Taller 3, `X` elige la casilla 3 con valor +1: crea una doble amenaza (líneas 3‑4‑5 y 0‑4‑8) que `O` no puede bloquear.
   - Llama la atención que no elige la casilla 8, que **gana de inmediato**. Como la utilidad solo distingue +1, 0 y −1, ganar ahora o dentro de dos turnos vale lo mismo, y se queda con la primera jugada de valor +1 que encuentra. Para preferir las victorias rápidas se puede usar una utilidad que dependa de la profundidad, por ejemplo `+(10 - profundidad)` para las victorias de `X`.
5. **El orden importa.** Con el orden centro → esquinas → bordes, Alfa–Beta pasa de 18 297 a 7 275 nodos (≈60 % menos) con el mismo valor. Es lo mismo que se vio en la sección 10: examinar primero las jugadas prometedoras ajusta antes α y β y produce más podas.

### Uso de IA generativa

* Se utilizó IA generativa con Claude Code como apoyo para la implementación de Alfa–Beta en Tres en raya Y la comparación con Minimax. Su propósito fue verificar el funcionamiento de las podas y entender el efecto del orden de exploración.